# Fashion-IQ Composed Image Retrieval

## Install package

Install OpenMP and GPU FAISS for hard-negative mining.


In [ ]:
!apt install -y libomp-dev
!pip install faiss-gpu-cu12


## Reproducibility seed

In [ ]:
import numpy as np
import torch
import random
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


## Unpack dataset

Unzip fashionIQ dataset. Replace the `dataset path here` with your dataset zipped file. Replace the `root` with the destination folder

In [ ]:
## If you are using Google Colab, you can mount your drive to the notebook.

#from google.colab import drive
#drive.mount('/content/drive')

# If you are using 7zip, you can unzip the dataset here
!apt-get -qq install p7zip-full
!7z x "dataset path here" -o "destination"
!mkdir -p "root/data"
!mkdir -p "root/data/images"
!mkdir -p "root/data/captions"
!mkdir -p "root/data/image_splits"
!mkdir -p "root/data/checkpoints"
!unzip "captions path here" -d "root/data"
!unzip "image_splits path here" -d "root/data"
!tar -xvzf "images path here" -C "root/data"


## API keys

Hugging Face and Weights & Biases tokens.


In [ ]:
HF_TOKEN = "API here"
wandb_API = "API here"

## Config

Play around with the fine-tune setting


In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    fashioniq_root: str = "root/data"
    image_dir: str = "root/data/images"
    output_dir: str = "root/data/checkpoints"

    siglip_model_name: str = "google/siglip-so400m-patch14-384"
    embed_dim: int = 512
    num_visual_tokens: int = 160

    resnet_input_size: int = 224
    resnet_extract_layers: List[str] = field(
        default_factory=lambda: ['layer1', "layer2", "layer3"]
    )
    resnet_tokens_per_layer: int = 16
    freeze_resnet_through: str = "layer2"
    lr_resnet: float = 1e-5

    num_fusion_layers: int = 2
    num_fusion_query_tokens: int = 8
    fusion_num_heads: int = 8
    fusion_dropout: float = 0.1
    text_max_length: int = 64

    siglip_vision_unfreeze_layers: int = 2
    lr_siglip_vision: float = 1e-6

    siglip_text_unfreeze_layers: int = 2
    lr_siglip_text: float = 1e-6

    num_hard_negatives: int = 24
    num_neg_per_sample: int = 8
    mine_every_n_epochs: int = 1

    target_agg_num_heads: int = 4

    gallery_batch_size: int = 0

    triplet_margin: float = 0.3
    loss_lambda_triplet: float = 2.5
    loss_lambda_contrastive: float = 1.5
    temperature: float = 0.07
    label_smoothing: float = 0.1

    categories: List[str] = field(
        default_factory=lambda: ["dress", "shirt", "toptee"]
    )
    batch_size: int = 24
    num_workers: int = 2
    lr_projector: float = 1e-4
    lr_fusion: float = 1e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.08
    num_epochs: int = 25
    grad_accum_steps: int = 4
    max_grad_norm: float = 1.0
    fp16: bool = True
    seed: int = 42

    recall_ks: List[int] = field(default_factory=lambda: [10, 50])
    eval_every: int = 2


cfg = Config()

gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name = torch.cuda.get_device_name(0).lower()

if "a100" in gpu_name:
    cfg.batch_size = 40
    cfg.grad_accum_steps = 3
    cfg.siglip_vision_unfreeze_layers = 6
    cfg.siglip_text_unfreeze_layers = 3
    cfg.num_fusion_layers = 4
    cfg.gallery_batch_size = 128
elif "v100" in gpu_name or gpu_mem > 15:
    cfg.batch_size = 12
    cfg.grad_accum_steps = 8
    cfg.siglip_vision_unfreeze_layers = 4
    cfg.siglip_text_unfreeze_layers = 2
    cfg.gallery_batch_size = 80
else:
    cfg.batch_size = 10
    cfg.grad_accum_steps = 10
    cfg.siglip_vision_unfreeze_layers = 3
    cfg.siglip_text_unfreeze_layers = 2
    cfg.gallery_batch_size = 48

if cfg.gallery_batch_size == 0:
    cfg.gallery_batch_size = max(cfg.batch_size * 4, 32)

eff_bs = cfg.batch_size * cfg.grad_accum_steps
print(f"  Model:     {cfg.siglip_model_name}")
print(f"  Query residual: ON")
print(f"  Text augmentation: ON")
print(f"  ResNet-50: layers={cfg.resnet_extract_layers}")
print(f"  Vision unfreeze: last {cfg.siglip_vision_unfreeze_layers} layers")
print(f"  Text unfreeze:   last {cfg.siglip_text_unfreeze_layers} layers")
print(f"  Batch: {cfg.batch_size} x {cfg.grad_accum_steps} = {eff_bs}")
print(f"  Loss: temp={cfg.temperature}, margin={cfg.triplet_margin}, "
      f"loss_trip={cfg.loss_lambda_triplet}, loss_cont={cfg.loss_lambda_contrastive}, "
      f"LS={cfg.label_smoothing}")
print(f"  Gallery batch (inference): {cfg.gallery_batch_size}")
print(f"  Mining: every {cfg.mine_every_n_epochs} epoch ")
print(f"  GPU: {gpu_name} ({gpu_mem:.0f}GB)")


## Dataset pre-process

In [ ]:
import json
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms as T
from transformers import SiglipProcessor


def get_resnet_transform(input_size: int = 224):
    return T.Compose([
        T.Resize((input_size, input_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225]),
    ])


class FashionIQDataset(Dataset):
    def __init__(
        self,
        root_dir: str,
        image_dir: str,
        categories: List[str],
        split: str = "train",
        processor=None,
        resnet_transform=None,
    ):
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.resnet_transform = resnet_transform
        self.text_augment = (split == "train")
        self.triplets = []
        self.base_idx = 0
        for cat in categories:
            cap_file = Path(root_dir) / "captions" / f"cap.{cat}.{split}.json"
            with open(cap_file) as f:
                for ann in json.load(f):
                    self.triplets.append((
                        ann["candidate"], ann["target"],
                        ann["captions"], cat
                    ))

    def _load_image(self, image_id: str) -> Image.Image:
        for ext in [".jpg", ".png", ".jpeg", ""]:
            path = self.image_dir / f"{image_id}{ext}"
            if path.exists():
                return Image.open(path).convert("RGB")
        raise FileNotFoundError(f"Image not found: {image_id}")

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        ref_id, target_id, captions, category = self.triplets[idx]
        ref_image = self._load_image(ref_id)
        target_image = self._load_image(target_id)

        if self.text_augment:
            caps = list(captions)
            if random.random() < 0.5:
                caps = caps[::-1]
            connectors = [". ", " and ", ". Also, ",
                          ". Additionally, ", ", and also "]
            conn = random.choice(connectors)
            if random.random() < 0.1 and len(caps) > 1:
                mod_text = random.choice(caps).strip()
            else:
                mod_text = f"{caps[0].strip()}{conn}{caps[1].strip()}"
        else:
            mod_text = f"{captions[0].strip()}. {captions[1].strip()}"

        ref_inputs = self.processor(images=ref_image, return_tensors="pt")
        target_inputs = self.processor(images=target_image, return_tensors="pt")

        out = {
            "ref_pixel_values": ref_inputs["pixel_values"].squeeze(0),
            "target_pixel_values": target_inputs["pixel_values"].squeeze(0),
            "mod_text": mod_text,
            "ref_id": ref_id,
            "target_id": target_id,
            "category": category,
            "idx": self.base_idx + idx,
        }
        if self.resnet_transform is not None:
            out["ref_resnet_pixels"] = self.resnet_transform(ref_image)
            out["target_resnet_pixels"] = self.resnet_transform(target_image)
        return out


class FashionIQEvalDataset(Dataset):
    def __init__(self, image_dir, image_ids, processor, resnet_transform=None):
        self.image_dir = Path(image_dir)
        self.image_ids = image_ids
        self.processor = processor
        self.resnet_transform = resnet_transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        for ext in [".jpg", ".png", ".jpeg", ""]:
            path = self.image_dir / f"{img_id}{ext}"
            if path.exists():
                img = Image.open(path).convert("RGB")
                inputs = self.processor(images=img, return_tensors="pt")
                siglip_px = inputs["pixel_values"].squeeze(0)
                if self.resnet_transform is not None:
                    resnet_px = self.resnet_transform(img)
                    return siglip_px, resnet_px, img_id
                else:
                    return siglip_px, img_id
        raise FileNotFoundError(f"Not found: {img_id}")


def collate_fn(batch):
    out = {
        "ref_pixel_values": torch.stack([s["ref_pixel_values"] for s in batch]),
        "target_pixel_values": torch.stack([s["target_pixel_values"] for s in batch]),
        "mod_texts": [s["mod_text"] for s in batch],
        "ref_ids": [s["ref_id"] for s in batch],
        "target_ids": [s["target_id"] for s in batch],
        "categories": [s["category"] for s in batch],
        "indices": [s["idx"] for s in batch],
    }
    if "ref_resnet_pixels" in batch[0]:
        out["ref_resnet_pixels"] = torch.stack(
            [s["ref_resnet_pixels"] for s in batch])
        out["target_resnet_pixels"] = torch.stack(
            [s["target_resnet_pixels"] for s in batch])
    return out


print(" Dataset (text augmentation + dual preprocessing)")


import torch


## ResNetLowLevelExtractor



In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torchvision.models import resnet50, ResNet50_Weights
from transformers import SiglipModel, AutoProcessor

class ResNetLowLevelExtractor(nn.Module):
    LAYER_CHANNELS = {
        "layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048
    }

    def __init__(self, output_dim, extract_layers=("layer2", "layer3"),
                 tokens_per_layer=16, freeze_through="layer2"):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.extract_layers = list(extract_layers)
        self.tokens_per_layer = tokens_per_layer
        pool_side = int(tokens_per_layer ** 0.5)
        assert pool_side * pool_side == tokens_per_layer
        self.adaptive_pool = nn.AdaptiveAvgPool2d((pool_side, pool_side))

        self.token_projectors = nn.ModuleDict({
            name: nn.Sequential(
                nn.LayerNorm(self.LAYER_CHANNELS[name]),
                nn.Linear(self.LAYER_CHANNELS[name], output_dim),
                nn.GELU(),
                nn.Linear(output_dim, output_dim),
            ) for name in extract_layers
        })
        pooled_dim = sum(self.LAYER_CHANNELS[n] for n in extract_layers)
        self.pool_projector = nn.Sequential(
            nn.LayerNorm(pooled_dim),
            nn.Linear(pooled_dim, output_dim),
            nn.GELU(),
        )
        self.pooled_input_dim = pooled_dim
        self._apply_freezing(freeze_through)

    def _apply_freezing(self, freeze_through):
        freeze_order = ["stem", "layer1", "layer2", "layer3", "layer4"]
        if freeze_through not in freeze_order:
            return
        stop = freeze_order.index(freeze_through)
        frozen, total = 0, 0
        for i, name in enumerate(freeze_order):
            for p in getattr(self, name).parameters():
                total += p.numel()
                if i <= stop:
                    p.requires_grad = False
                    frozen += p.numel()
        print(f"   ResNet: froze stem→{freeze_through} "
              f"({frozen/1e6:.1f}M / {total/1e6:.1f}M)")

    def _backbone(self, x):
        feats = {}
        x = self.stem(x)
        x = self.layer1(x)
        if "layer1" in self.extract_layers: feats["layer1"] = x
        x = self.layer2(x)
        if "layer2" in self.extract_layers: feats["layer2"] = x
        x = self.layer3(x)
        if "layer3" in self.extract_layers: feats["layer3"] = x
        if "layer4" in self.extract_layers:
            x = self.layer4(x); feats["layer4"] = x
        return feats

    def _feats_to_tokens(self, feats):
        all_tokens = []
        for name in self.extract_layers:
            fm = self.adaptive_pool(feats[name])
            tokens = fm.flatten(2).permute(0, 2, 1)
            tokens = self.token_projectors[name](tokens)
            all_tokens.append(tokens)
        return torch.cat(all_tokens, dim=1)

    def _feats_to_pooled(self, feats):
        parts = [F.adaptive_avg_pool2d(feats[n], 1).flatten(1)
                 for n in self.extract_layers]
        return self.pool_projector(torch.cat(parts, dim=-1))

    def forward(self, x):
        return self._feats_to_tokens(self._backbone(x))

    def forward_pooled(self, x):
        return self._feats_to_pooled(self._backbone(x))

    def forward_both(self, x):
        feats = self._backbone(x)
        return self._feats_to_tokens(feats), self._feats_to_pooled(feats)


## Visual modules


In [ ]:
class VisualTokenCompressor(nn.Module):
    def __init__(self, dim, num_output_tokens=64, num_heads=8):
        super().__init__()
        self.queries = nn.Parameter(
            torch.randn(1, num_output_tokens, dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(
            dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))

    def forward(self, patch_tokens):
        B = patch_tokens.shape[0]
        q = self.queries.expand(B, -1, -1)
        attended, _ = self.cross_attn(q, patch_tokens, patch_tokens)
        x = self.norm1(q + attended)
        return self.norm2(x + self.ffn(x))


class MultiScaleVisualMerger(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.siglip_type = nn.Parameter(torch.zeros(1, 1, dim))
        self.resnet_type = nn.Parameter(torch.zeros(1, 1, dim))
        self.norm = nn.LayerNorm(dim)

    def forward(self, siglip_tokens, resnet_tokens):
        s = siglip_tokens + self.siglip_type
        r = resnet_tokens + self.resnet_type
        return self.norm(torch.cat([s, r], dim=1))


class TargetTokenAggregator(nn.Module):
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(
            dim, num_heads, batch_first=True, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(dim * 2, dim))
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, tokens):
        B = tokens.shape[0]
        q = self.query.expand(B, -1, -1)
        attended, _ = self.cross_attn(q, tokens, tokens)
        x = self.norm1(q + attended)
        x = self.norm2(x + self.ffn(x))
        return x.squeeze(1)


class EnhancedTargetFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.resnet_gate = nn.Sequential(
            nn.Linear(dim * 2, dim), nn.Sigmoid())
        self.token_gate = nn.Sequential(
            nn.Linear(dim * 2, dim), nn.Sigmoid())
        self.norm = nn.LayerNorm(dim)

    def forward(self, siglip_pooled, resnet_pooled=None,
                token_aggregated=None):
        x = siglip_pooled
        if resnet_pooled is not None:
            g = self.resnet_gate(torch.cat([x, resnet_pooled], dim=-1))
            x = x + g * resnet_pooled
        if token_aggregated is not None:
            g = self.token_gate(torch.cat([x, token_aggregated], dim=-1))
            x = x + g * token_aggregated
        return self.norm(x)


class ResidualBlend(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
            nn.Sigmoid(),
        )
        nn.init.constant_(self.gate[0].bias, -0.85)

    def forward(self, composed, reference):
        alpha = self.gate(torch.cat([composed, reference], dim=-1))
        return F.normalize(composed + alpha * reference, dim=-1)


class CompositionFusion(nn.Module):
    def __init__(self, dim, num_heads=8, num_layers=2,
                 num_query_tokens=8, dropout=0.1):
        super().__init__()
        self.query_tokens = nn.Parameter(
            torch.randn(1, num_query_tokens, dim) * 0.02)
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(nn.ModuleList([
                nn.MultiheadAttention(dim, num_heads,
                                      batch_first=True, dropout=dropout),
                nn.LayerNorm(dim),
                nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(),
                              nn.Dropout(dropout), nn.Linear(dim * 4, dim)),
                nn.LayerNorm(dim),
            ]))
        self.final_norm = nn.LayerNorm(dim)

    def forward(self, visual_tokens, text_tokens, text_attention_mask=None):
        B = visual_tokens.shape[0]
        device = visual_tokens.device
        context = torch.cat([visual_tokens, text_tokens], dim=1)
        if text_attention_mask is not None:
            vis_mask = torch.ones(B, visual_tokens.shape[1],
                                  device=device,
                                  dtype=text_attention_mask.dtype)
            kpm = ~torch.cat([vis_mask, text_attention_mask], dim=1).bool()
        else:
            kpm = None
        x = self.query_tokens.expand(B, -1, -1)
        for cross_attn, n1, ffn, n2 in self.layers:
            att, _ = cross_attn(x, context, context, key_padding_mask=kpm)
            x = n1(x + att)
            x = n2(x + ffn(x))
        return self.final_norm(x.mean(dim=1))


class TargetProjector(nn.Module):
    def __init__(self, dim, embed_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, embed_dim * 2),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(embed_dim * 2, embed_dim))
    def forward(self, x):
        return F.normalize(self.proj(x), dim=-1)

class QueryProjector(nn.Module):
    def __init__(self, dim, embed_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, embed_dim * 2),
            nn.GELU(), nn.Dropout(0.1),
            nn.Linear(embed_dim * 2, embed_dim, bias=False))
    def forward(self, x):
        return F.normalize(self.proj(x), dim=-1)


## SigLIP layer helpers

In [ ]:
def _get_vision_encoder_layers(vision_model):
    vm = vision_model
    for attr in ["vision_model", "model"]:
        if hasattr(vm, attr):
            vm = getattr(vm, attr); break
    if hasattr(vm, "encoder") and hasattr(vm.encoder, "layers"):
        return vm.encoder.layers
    raise AttributeError(f"Cannot find encoder layers in {type(vision_model)}")

def _get_text_encoder_layers(text_model):
    tm = text_model
    for attr in ["text_model", "model"]:
        if hasattr(tm, attr):
            tm = getattr(tm, attr); break
    if hasattr(tm, "encoder") and hasattr(tm.encoder, "layers"):
        return tm.encoder.layers
    raise AttributeError(f"Cannot find encoder layers in {type(text_model)}")

def _get_vision_head_params(vision_model):
    params = []
    vm = getattr(vision_model, "vision_model",
                 getattr(vision_model, "model", vision_model))
    for attr in ["post_layernorm", "layernorm", "head",
                 "multi_head_attention_pooling_head"]:
        if hasattr(vm, attr):
            params.extend(getattr(vm, attr).parameters())
    return params

def _get_text_head_params(text_model):
    tm = getattr(text_model, "text_model",
                 getattr(text_model, "model", text_model))
    for attr in ["final_layer_norm", "layernorm", "head", "norm"]:
        if hasattr(tm, attr):
            return list(getattr(tm, attr).parameters())
    return []


## CIRModel

SigLIP vision + text, always-on ResNet, compressor, aggregator, fusion, residual.



In [ ]:
class CIRModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        print(f"Loading SigLIP: {cfg.siglip_model_name} ")
        base_model = SiglipModel.from_pretrained(cfg.siglip_model_name)
        self.siglip_vision = base_model.vision_model
        self.siglip_text = base_model.text_model

        vision_cfg = base_model.config.vision_config
        text_cfg = base_model.config.text_config
        siglip_dim = vision_cfg.hidden_size
        num_vis_layers = vision_cfg.num_hidden_layers
        num_txt_layers = text_cfg.num_hidden_layers
        ps = vision_cfg.patch_size
        im = vision_cfg.image_size
        if isinstance(im, (list, tuple)):
            im = im[0]
        num_patches = (im // ps) ** 2
        print(f"  dim={siglip_dim}, vis_layers={num_vis_layers}, "
              f"patches={num_patches}, txt_layers={num_txt_layers}")

        self.siglip_processor = AutoProcessor.from_pretrained(
            cfg.siglip_model_name)
        self.tokenizer = self.siglip_processor.tokenizer

        print("Loading ResNet-50")
        self.resnet = ResNetLowLevelExtractor(
            output_dim=siglip_dim,
            extract_layers=cfg.resnet_extract_layers,
            tokens_per_layer=cfg.resnet_tokens_per_layer,
            freeze_through=cfg.freeze_resnet_through)
        num_resnet_tokens = (
            len(cfg.resnet_extract_layers) * cfg.resnet_tokens_per_layer)
        print(f"   ResNet tokens: {num_resnet_tokens}")
        self.visual_merger = MultiScaleVisualMerger(siglip_dim)

        self.target_aggregator = TargetTokenAggregator(
            siglip_dim, num_heads=cfg.target_agg_num_heads,
            dropout=cfg.fusion_dropout)
        print(f"   Target aggregator: 1 query × "
              f"{num_patches + num_resnet_tokens} kv → 1 vec")

        self.target_fusion = EnhancedTargetFusion(siglip_dim)
        print(f"   Target fusion: 3-way gated "
              f"(resnet=True, agg=True)")

        self._setup_vision_freezing(cfg, num_vis_layers)
        self._setup_text_freezing(cfg, num_txt_layers)

        try:
            self.siglip_vision.gradient_checkpointing_enable()
            print("   Grad checkpointing: ON (native)")
        except Exception:
            try:
                vm = getattr(self.siglip_vision, "vision_model",
                             self.siglip_vision)
                vm.encoder.gradient_checkpointing = True
                print("   Grad checkpointing: ON (manual)")
            except Exception as e:
                print(f"     Grad checkpointing failed: {e}")

        self.visual_compressor = VisualTokenCompressor(
            siglip_dim, cfg.num_visual_tokens,
            num_heads=8 if siglip_dim >= 1024 else 4)
        print(f"   Compressor: {num_patches} → "
              f"{cfg.num_visual_tokens} tokens")

        self.fusion = CompositionFusion(
            dim=siglip_dim, num_heads=cfg.fusion_num_heads,
            num_layers=cfg.num_fusion_layers,
            num_query_tokens=cfg.num_fusion_query_tokens,
            dropout=cfg.fusion_dropout)
        total_vis = cfg.num_visual_tokens + num_resnet_tokens
        print(f"   Query fusion: {total_vis} visual + text tokens")

        self.query_projector = QueryProjector(siglip_dim, cfg.embed_dim)
        self.target_projector = TargetProjector(siglip_dim, cfg.embed_dim)

        self.residual_blend = ResidualBlend(cfg.embed_dim)
        object.__setattr__(self, "reference_fusion", self.target_fusion)
        object.__setattr__(self, "reference_projector", self.target_projector)
        print("   Query residual: ON (detach_ref=True)")

        self._text_templates = [
            "The reference image should be modified: {}",
            "Change the image so that: {}",
            "Modify to: {}",
            "Make these changes: {}",
            "Update the reference: {}",
        ]
        self._default_template = \
            "The reference image should be modified: {}"

        self._print_param_summary()

    def _setup_vision_freezing(self, cfg, num_layers):
        n = cfg.siglip_vision_unfreeze_layers
        for p in self.siglip_vision.parameters():
            p.requires_grad = False
        layers = _get_vision_encoder_layers(self.siglip_vision)
        for i in range(num_layers - n, num_layers):
            for p in layers[i].parameters():
                p.requires_grad = True
        for p in _get_vision_head_params(self.siglip_vision):
            p.requires_grad = True
        uf = sum(p.numel() for p in self.siglip_vision.parameters()
                 if p.requires_grad)
        tot = sum(p.numel() for p in self.siglip_vision.parameters())
        print(f"   SigLIP Vision: unfroze last {n}/{num_layers} "
              f"({uf/1e6:.1f}M / {tot/1e6:.1f}M)")

    def _setup_text_freezing(self, cfg, num_layers):
        n = cfg.siglip_text_unfreeze_layers
        for p in self.siglip_text.parameters():
            p.requires_grad = False
        layers = _get_text_encoder_layers(self.siglip_text)
        for i in range(num_layers - n, num_layers):
            for p in layers[i].parameters():
                p.requires_grad = True
        for p in _get_text_head_params(self.siglip_text):
            p.requires_grad = True
        uf = sum(p.numel() for p in self.siglip_text.parameters()
                 if p.requires_grad)
        tot = sum(p.numel() for p in self.siglip_text.parameters())
        print(f"   SigLIP Text: unfroze last {n}/{num_layers} "
              f"({uf/1e6:.1f}M / {tot/1e6:.1f}M)")

    def _print_param_summary(self):
        groups = {
            "SigLIP Vision (frozen)": ([], []),
            "SigLIP Vision (train)":  ([], []),
            "SigLIP Text (frozen)":   ([], []),
            "SigLIP Text (train)":    ([], []),
            "ResNet-50 (frozen)":     ([], []),
            "ResNet-50 (train)":      ([], []),
            "Visual Compressor":      ([], []),
            "Visual Merger":          ([], []),
            "Target Aggregator":      ([], []),
            "Target Fusion":          ([], []),
            "Residual Blend":         ([], []),
            "Fusion Module":          ([], []),
            "Query Projector":        ([], []),
            "Target Projector":       ([], []),
            "Other":                  ([], []),
        }
        for name, p in self.named_parameters():
            if "siglip_vision" in name:
                k = ("SigLIP Vision (train)" if p.requires_grad
                     else "SigLIP Vision (frozen)")
            elif "siglip_text" in name:
                k = ("SigLIP Text (train)" if p.requires_grad
                     else "SigLIP Text (frozen)")
            elif "resnet." in name:
                k = ("ResNet-50 (train)" if p.requires_grad
                     else "ResNet-50 (frozen)")
            elif "visual_compressor" in name: k = "Visual Compressor"
            elif "visual_merger" in name:     k = "Visual Merger"
            elif "target_aggregator" in name: k = "Target Aggregator"
            elif "target_fusion" in name:     k = "Target Fusion"
            elif "residual_blend" in name:    k = "Residual Blend"
            elif "fusion" in name:            k = "Fusion Module"
            elif "query_projector" in name:   k = "Query Projector"
            elif "target_projector" in name:  k = "Target Projector"
            else:                             k = "Other"
            (groups[k][0] if p.requires_grad else groups[k][1]).append(
                p.numel())

        print(f"\n  {'Component':<28} {'Trainable':>12} {'Frozen':>12}")
        print(f"  {'─'*28} {'─'*12} {'─'*12}")
        tt, tf = 0, 0
        for key, (train, frozen) in groups.items():
            t, f_ = sum(train), sum(frozen)
            if t > 0 or f_ > 0:
                print(f"  {key:<28} {t/1e6:>10.2f}M {f_/1e6:>10.2f}M")
            tt += t; tf += f_
        print(f"  {'─'*28} {'─'*12} {'─'*12}")
        print(f"  {'TOTAL':<28} {tt/1e6:>10.2f}M {tf/1e6:>10.2f}M")
        print(f"  Trainable: {tt/1e6:.2f}M ({tt/(tt+tf)*100:.1f}%)\n")
        print(f"  Est. optimizer VRAM: {tt * 16 / 1e9:.2f} GB")

    def _get_pooled(self, model_output, hidden_states):
        if (hasattr(model_output, 'pooler_output')
                and model_output.pooler_output is not None):
            return model_output.pooler_output.float()
        return hidden_states.mean(dim=1)

    def _extract_siglip_features(self, pixel_values, compress=True):
        out = self.siglip_vision(pixel_values=pixel_values)
        patch_tokens = out.last_hidden_state.float()
        pooled = self._get_pooled(out, patch_tokens)
        if compress:
            if self.training:
                patch_tokens = checkpoint(
                    self.visual_compressor, patch_tokens,
                    use_reentrant=False)
            else:
                patch_tokens = self.visual_compressor(patch_tokens)
        return patch_tokens, pooled

    def _extract_text_features(self, texts, device):
        tok = self.tokenizer(
            texts, padding=True, truncation=True,
            max_length=self.cfg.text_max_length,
            return_tensors="pt").to(device)
        out = self.siglip_text(
            input_ids=tok["input_ids"],
            attention_mask=tok.get("attention_mask"))
        return out.last_hidden_state.float(), tok.get("attention_mask")

    def _merge_visual(self, siglip_tokens, resnet_tokens):
        if self.visual_merger is not None and resnet_tokens is not None:
            return self.visual_merger(siglip_tokens, resnet_tokens)
        return siglip_tokens

    @torch.no_grad()
    def encode_target(self, pixel_values, resnet_pixels=None):
        out = self.siglip_vision(pixel_values=pixel_values)
        siglip_patches = out.last_hidden_state.float()
        siglip_pooled = self._get_pooled(out, siglip_patches)

        resnet_pooled, resnet_tokens = None, None
        if resnet_pixels is not None:
            resnet_tokens, resnet_pooled = \
                self.resnet.forward_both(resnet_pixels)

        if resnet_tokens is not None:
            merged = self._merge_visual(siglip_patches, resnet_tokens)
        else:
            merged = siglip_patches
        token_agg = self.target_aggregator(merged)

        fused = self.target_fusion(siglip_pooled, resnet_pooled, token_agg)
        return self.target_projector(fused)

    @torch.no_grad()
    def encode_target_fast(self, pixel_values, resnet_pixels=None):
        out = self.siglip_vision(pixel_values=pixel_values)
        patches = out.last_hidden_state.float()
        siglip_pooled = self._get_pooled(out, patches)

        resnet_pooled = None
        if resnet_pixels is not None:
            resnet_pooled = self.resnet.forward_pooled(resnet_pixels)

        fused = self.target_fusion(siglip_pooled, resnet_pooled, None)
        return self.target_projector(fused)

    def encode_target_train(self, pixel_values, resnet_pixels=None):
        siglip_patches, siglip_pooled = self._extract_siglip_features(
            pixel_values, compress=False)

        resnet_pooled, resnet_tokens = None, None
        if resnet_pixels is not None:
            resnet_tokens, resnet_pooled = \
                self.resnet.forward_both(resnet_pixels)

        if self.target_aggregator is not None:  
            if resnet_tokens is not None:
                merged = self._merge_visual(siglip_patches, resnet_tokens)
            else:
                merged = siglip_patches
            token_agg = self.target_aggregator(merged)

            fused = self.target_fusion(siglip_pooled, resnet_pooled, token_agg)
            return self.target_projector(fused)

    def encode_query(self, ref_pixel_values, mod_texts,
                     resnet_pixels=None):
        device = ref_pixel_values.device

        siglip_tokens, siglip_pooled = self._extract_siglip_features(
            ref_pixel_values, compress=True)

        resnet_tokens, resnet_pooled = None, None
        if resnet_pixels is not None:
            resnet_tokens, resnet_pooled = \
                self.resnet.forward_both(resnet_pixels)
        visual_tokens = self._merge_visual(siglip_tokens, resnet_tokens)

        if self.training and hasattr(self, '_text_templates'):
            texts = [random.choice(self._text_templates).format(t)
                     for t in mod_texts]
        else:
            texts = [self._default_template.format(t)
                     for t in mod_texts]
        text_tokens, text_mask = self._extract_text_features(texts, device)

        fused = self.fusion(visual_tokens, text_tokens, text_mask)
        composed_emb = self.query_projector(fused)

        if self.residual_blend is not None:
            ref_fused = self.reference_fusion(
                siglip_pooled, resnet_pooled, None)
            ref_emb = self.reference_projector(ref_fused).detach()
            return self.residual_blend(composed_emb, ref_emb)
        return composed_emb

    def get_trainable_param_groups(self):
        proj_params = (
            list(self.query_projector.parameters()) +
            list(self.target_projector.parameters()) +
            list(self.target_fusion.parameters())
        )
        proj_params += list(self.visual_compressor.parameters())
        if self.visual_merger is not None:
            proj_params += list(self.visual_merger.parameters())
        proj_params += list(self.target_aggregator.parameters())
        proj_params += list(self.residual_blend.parameters())

        fusion_params = list(self.fusion.parameters())

        resnet_params = [p for p in self.resnet.parameters()
                         if p.requires_grad]

        siglip_vision_params = [
            p for p in self.siglip_vision.parameters() if p.requires_grad]
        siglip_text_params = [
            p for p in self.siglip_text.parameters() if p.requires_grad]

        groups = [
            {"params": proj_params,
             "lr": self.cfg.lr_projector, "weight_decay": 0.01},
            {"params": fusion_params,
             "lr": self.cfg.lr_fusion, "weight_decay": 0.01},
        ]
        if resnet_params:
            groups.append({"params": resnet_params,
                           "lr": self.cfg.lr_resnet, "weight_decay": 0.01})
        if siglip_vision_params:
            groups.append({"params": siglip_vision_params,
                           "lr": self.cfg.lr_siglip_vision,
                           "weight_decay": 0.01})
        if siglip_text_params:
            groups.append({"params": siglip_text_params,
                           "lr": self.cfg.lr_siglip_text,
                           "weight_decay": 0.01})
        return groups


print(" CIRModel is ready")


## Embedding cache and hard-negative miner


In [ ]:
import time
import numpy as np
import faiss
import gc
from torch.cuda.amp import autocast
from tqdm.notebook import tqdm


class EmbeddingCache:
    def __init__(self):
        self.embeddings = None
        self.id_list = []
        self.id2idx = {}
        self.id2cat = {}
        self.cat2indices = {}

    @torch.no_grad()
    def build(self, model, gallery_loader, device,
              id2cat=None):
        print(" Building embedding cache [FAST (pooled-only)]")
        t0 = time.time()
        model.eval()
        all_embs, all_ids = [], []

        encode_fn = model.encode_target_fast

        for batch in tqdm(gallery_loader, desc="Encoding gallery"):
            if len(batch) == 3:
                siglip_px, resnet_px, ids = batch
                resnet_px = resnet_px.to(device)
            else:
                siglip_px, ids = batch
                resnet_px = None

            with autocast():
                emb = encode_fn(siglip_px.to(device), resnet_px)
            all_embs.append(emb.float().cpu())
            if isinstance(ids, torch.Tensor):
                ids = ids.tolist()
            all_ids.extend(ids)

        self.embeddings = torch.cat(all_embs, 0)
        self.id_list = all_ids
        self.id2idx = {iid: i for i, iid in enumerate(all_ids)}

        self.id2cat = id2cat or {}
        self.cat2indices = {}
        if self.id2cat:
            for img_id, idx in self.id2idx.items():
                cat = self.id2cat.get(img_id)
                if cat is not None:
                    self.cat2indices.setdefault(cat, []).append(idx)
            dist = ", ".join(f"{c}={len(v)}"
                             for c, v in sorted(self.cat2indices.items()))
            print(f"   Category distribution: {dist}")

        elapsed = time.time() - t0
        mem_mb = self.embeddings.nelement() * 4 / 1e6
        speed = len(all_ids) / max(elapsed, 0.1)
        print(f" Cached {len(all_ids)} embeddings ({mem_mb:.1f} MB) "
              f"in {elapsed:.1f}s ({speed:.0f} img/s)")
        return self

    def lookup_indices(self, indices):
        return self.embeddings[indices]

    def get_numpy_normalized(self):
        emb_np = self.embeddings.numpy().astype(np.float32).copy()
        faiss.normalize_L2(emb_np)
        return emb_np

    def get_category_numpy_normalized(self, category):
        indices = self.cat2indices.get(category, [])
        if not indices:
            return None, []
        emb = self.embeddings[indices].numpy().astype(np.float32).copy()
        faiss.normalize_L2(emb)
        return emb, indices


class HardNegativeMiner:
    def __init__(self, num_hard_negatives=16, category_aware=True):
        self.num_hard_negatives = num_hard_negatives
        self.category_aware = category_aware

    def mine(self, cache, triplets):
        if self.category_aware and cache.cat2indices:
            return self._mine_category_aware(cache, triplets)
        return self._mine_global(cache, triplets)

    def _mine_category_aware(self, cache, triplets):
        print("  Mining category-aware hard negatives")
        search_k = self.num_hard_negatives + 50
        cat_faiss, cat_local_map = {}, {}

        for cat, cache_idxs in cache.cat2indices.items():
            emb_np, _ = cache.get_category_numpy_normalized(cat)
            if emb_np is None or len(emb_np) == 0:
                continue
            D = emb_np.shape[1]
            index = faiss.IndexFlatIP(D)
            if faiss.get_num_gpus() > 0:
                res = faiss.StandardGpuResources()
                index = faiss.index_cpu_to_gpu(res, 0, index)
            index.add(emb_np)
            cat_faiss[cat] = (index, cache_idxs)
            cat_local_map[cat] = {ci: li for li, ci in enumerate(cache_idxs)}
            print(f"   {cat}: {len(cache_idxs)} items indexed")

        cat_triplets = {}
        for sidx, (rid, tid, _, cat) in enumerate(triplets):
            cat_triplets.setdefault(cat, []).append((sidx, rid, tid))

        hard_neg_map = {}
        for cat, trips in cat_triplets.items():
            if cat not in cat_faiss:
                continue
            faiss_idx, cache_idxs = cat_faiss[cat]
            c2l = cat_local_map[cat]

            target_to_samples = {}
            for sidx, rid, tid in trips:
                target_to_samples.setdefault(tid, []).append((sidx, rid))

            valid_targets, valid_local = [], []
            for tid in target_to_samples:
                ci = cache.id2idx.get(tid)
                if ci is not None and ci in c2l:
                    valid_targets.append(tid)
                    valid_local.append(c2l[ci])

            if not valid_local:
                continue

            emb_np, _ = cache.get_category_numpy_normalized(cat)
            query_vecs = emb_np[valid_local]
            _, nn_local = faiss_idx.search(query_vecs, search_k)

            for i, tid in enumerate(valid_targets):
                exclude_local = set()
                ci_target = cache.id2idx[tid]
                if ci_target in c2l:
                    exclude_local.add(c2l[ci_target])
                for sidx, rid in target_to_samples[tid]:
                    ci_ref = cache.id2idx.get(rid)
                    if ci_ref is not None and ci_ref in c2l:
                        exclude_local.add(c2l[ci_ref])

                negs = []
                for j in range(search_k):
                    li = int(nn_local[i, j])
                    if li >= 0 and li not in exclude_local:
                        negs.append(cache_idxs[li])
                    if len(negs) >= self.num_hard_negatives:
                        break
                if negs:
                    for sidx, _ in target_to_samples[tid]:
                        hard_neg_map[sidx] = negs

        avg_n = (np.mean([len(v) for v in hard_neg_map.values()])
                 if hard_neg_map else 0)
        print(f"⛏️  Mined for {len(hard_neg_map)} samples "
              f"(avg {avg_n:.1f} negs)")
        return hard_neg_map

    def _mine_global(self, cache, triplets):
        print("  Mining hard negatives (global)")
        emb_np = cache.get_numpy_normalized()
        D = emb_np.shape[1]
        index = faiss.IndexFlatIP(D)
        if faiss.get_num_gpus() > 0:
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
        index.add(emb_np)

        search_k = self.num_hard_negatives + 50
        target_to_samples = {}
        for sidx, (rid, tid, _, _) in enumerate(triplets):
            target_to_samples.setdefault(tid, []).append((sidx, rid))

        valid_targets, valid_idxs = [], []
        for tid in target_to_samples:
            if tid in cache.id2idx:
                valid_targets.append(tid)
                valid_idxs.append(cache.id2idx[tid])
        if not valid_idxs:
            return {}

        query_vecs = emb_np[valid_idxs]
        _, nn_indices = index.search(query_vecs, search_k)

        hard_neg_map = {}
        for i, tid in enumerate(valid_targets):
            exclude = {cache.id2idx[tid]}
            for sidx, rid in target_to_samples[tid]:
                if rid in cache.id2idx:
                    exclude.add(cache.id2idx[rid])
            negs = []
            for j in range(search_k):
                ni = int(nn_indices[i, j])
                if ni >= 0 and ni not in exclude:
                    negs.append(ni)
                if len(negs) >= self.num_hard_negatives:
                    break
            if negs:
                for sidx, _ in target_to_samples[tid]:
                    hard_neg_map[sidx] = negs

        avg_n = (np.mean([len(v) for v in hard_neg_map.values()])
                 if hard_neg_map else 0)
        print(f"⛏️  Mined for {len(hard_neg_map)} samples "
              f"(avg {avg_n:.1f} negs)")
        return hard_neg_map


print(" Cache + Category-Aware Miner ready")


## Losses


In [ ]:
class HardNegTripletLoss(nn.Module):
    def __init__(self, margin=0.3, top_k=8):
        super().__init__()
        self.margin = margin
        self.top_k = top_k

    def forward(self, query, target_pos, neg_embeds, neg_mask):
        d_pos = 1.0 - F.cosine_similarity(query, target_pos, dim=-1)
        d_neg = 1.0 - F.cosine_similarity(
            query.unsqueeze(1).expand_as(neg_embeds), neg_embeds, dim=-1)
        d_neg = d_neg.masked_fill(~neg_mask, float("inf"))

        K = min(self.top_k, d_neg.shape[1])
        d_neg_topk, _ = d_neg.topk(K, dim=1, largest=False)

        d_pos_exp = d_pos.unsqueeze(1).expand_as(d_neg_topk)
        losses = F.relu(d_pos_exp - d_neg_topk + self.margin)

        valid = d_neg_topk < float("inf")
        losses = (losses * valid.float()).sum(1) / \
                 valid.float().sum(1).clamp(min=1)
        return losses.mean()


class BatchContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07, label_smoothing=0.0):
        super().__init__()
        self.temperature = temperature
        self.label_smoothing = label_smoothing

    def forward(self, query, target):
        sim = query @ target.t() / self.temperature
        B = sim.shape[0]
        labels = torch.arange(B, device=sim.device)

        loss_q2t = F.cross_entropy(
            sim, labels, label_smoothing=self.label_smoothing)
        loss_t2q = F.cross_entropy(
            sim.t(), labels, label_smoothing=self.label_smoothing)

        return (loss_q2t + loss_t2q) / 2.0


class CIRLoss(nn.Module):
    def __init__(self, margin=0.3, temperature=0.07,
                 lam_triplet=1.0, lam_contrastive=1.0,
                 label_smoothing=0.0):
        super().__init__()
        self.triplet = HardNegTripletLoss(margin, top_k=8)
        self.contrastive = BatchContrastiveLoss(
            temperature, label_smoothing)
        self.lam_t = lam_triplet
        self.lam_c = lam_contrastive

    def forward(self, query, target, neg_embeds=None, neg_mask=None):
        l_c = self.contrastive(query, target)
        l_t = torch.tensor(0.0, device=query.device)
        if neg_embeds is not None and neg_mask is not None \
                and neg_mask.any():
            l_t = self.triplet(query, target, neg_embeds, neg_mask)
        total = self.lam_c * l_c + self.lam_t * l_t
        return {"total": total, "contrastive": l_c, "triplet": l_t}


print(" Losses (label smoothing + rebalanced)")


from torch.cuda.amp import autocast


## evaluation

In [ ]:
@torch.no_grad()
def evaluate_recall(model, query_loader, gallery_loaders,
                    recall_ks=[10, 50], device=torch.device("cuda")):
    model.eval()

    gal_embeds, gal_id2idx = {}, {}
    for cat, gl in gallery_loaders.items():
        embs, ids = [], []
        for batch in tqdm(gl, desc=f"Gallery {cat}", leave=False):
            if len(batch) == 3:
                siglip_px, resnet_px, img_ids = batch
                resnet_px = resnet_px.to(device)
            else:
                siglip_px, img_ids = batch
                resnet_px = None

            with autocast():
                embs.append(
                    model.encode_target(
                        siglip_px.to(device), resnet_px
                    ).float().cpu()
                )
            ids.extend(img_ids)
        gal_embeds[cat] = torch.cat(embs, 0)
        gal_id2idx[cat] = {iid: i for i, iid in enumerate(ids)}

    results = {f"{c}_R@{k}": [] for c in gal_embeds for k in recall_ks}

    for batch in tqdm(query_loader, desc="Eval", leave=False):
        ref_px = batch["ref_pixel_values"].to(device)
        resnet_px = batch.get("ref_resnet_pixels")
        if resnet_px is not None:
            resnet_px = resnet_px.to(device)

        with autocast():
            q = model.encode_query(
                ref_px, batch["mod_texts"], resnet_px
            ).float().cpu()

        for i in range(len(batch["target_ids"])):
            cat = batch["categories"][i]
            tid = batch["target_ids"][i]
            if tid not in gal_id2idx[cat]:
                continue
            sims = F.cosine_similarity(q[i:i+1], gal_embeds[cat], dim=-1)
            ref_id = batch["ref_ids"][i]
            if ref_id in gal_id2idx[cat]:
                sims[gal_id2idx[cat][ref_id]] = -1e9

            _, ranked = sims.sort(descending=True)
            rank = (ranked == gal_id2idx[cat][tid]).nonzero(as_tuple=True)[0].item()
            for k in recall_ks:
                results[f"{cat}_R@{k}"].append(1.0 if rank < k else 0.0)

    metrics = {}
    for key, vals in results.items():
        if vals:
            metrics[key] = np.mean(vals)
    for k in recall_ks:
        scores = [metrics.get(f"{c}_R@{k}", 0) for c in gal_embeds]
        metrics[f"avg_R@{k}"] = np.mean(scores)
    return metrics


print(" Evaluation (dual-pixel aware)")

## Training utilities


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class CategoryBatchSampler(torch.utils.data.Sampler):
    def __init__(self, concat_dataset, categories, batch_size,
                 drop_last=True):
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.cat_indices = {}
        offset = 0
        for ds, cat in zip(concat_dataset.datasets, categories):
            self.cat_indices.setdefault(cat, []).extend(
                range(offset, offset + len(ds)))
            offset += len(ds)
        for cat, idxs in self.cat_indices.items():
            n_batches = len(idxs) // batch_size if drop_last \
                        else (len(idxs) + batch_size - 1) // batch_size
            print(f"   {cat}: {len(idxs)} samples → {n_batches} batches")

    def __iter__(self):
        all_batches = []
        for cat, indices in self.cat_indices.items():
            shuffled = indices.copy()
            random.shuffle(shuffled)
            for i in range(0, len(shuffled), self.batch_size):
                batch = shuffled[i : i + self.batch_size]
                if len(batch) == self.batch_size:
                    all_batches.append(batch)
                elif not self.drop_last:
                    all_batches.append(batch)
        random.shuffle(all_batches)
        for batch in all_batches:
            yield batch

    def __len__(self):
        total = 0
        for indices in self.cat_indices.values():
            n = len(indices)
            if self.drop_last:
                total += n // self.batch_size
            else:
                total += (n + self.batch_size - 1) // self.batch_size
        return total


def build_id2cat_mapping(cfg, split="train"):
    id2cat = {}
    for cat in cfg.categories:
        path = (f"{cfg.fashioniq_root}/image_splits/"
                f"split.{cat}.{split}.json")
        with open(path) as f:
            for img_id in json.load(f):
                id2cat[img_id] = cat
    print(f"   id2cat: {len(id2cat)} images across "
          f"{len(cfg.categories)} categories ({split})")
    return id2cat


def build_gallery_loaders(cfg, processor, resnet_transform, split="val"):
    gbs = cfg.gallery_batch_size
    loaders = {}
    for cat in cfg.categories:
        with open(f"{cfg.fashioniq_root}/image_splits/"
                  f"split.{cat}.{split}.json") as f:
            ids = json.load(f)
        ds = FashionIQEvalDataset(
            cfg.image_dir, ids, processor,
            resnet_transform=resnet_transform)
        loaders[cat] = DataLoader(
            ds, batch_size=gbs, shuffle=False,
            num_workers=cfg.num_workers, pin_memory=True,
            persistent_workers=True if cfg.num_workers > 0 else False)
    return loaders


def build_full_gallery_loader(cfg, processor, resnet_transform,
                               split="train"):
    gbs = cfg.gallery_batch_size
    all_ids = set()
    for cat in cfg.categories:
        with open(f"{cfg.fashioniq_root}/image_splits/"
                  f"split.{cat}.{split}.json") as f:
            all_ids.update(json.load(f))
    ds = FashionIQEvalDataset(
        cfg.image_dir, list(all_ids), processor,
        resnet_transform=resnet_transform)
    n_batches = (len(ds) + gbs - 1) // gbs
    print(f"   Gallery loader: {len(ds)} imgs, bs={gbs}, "
          f"~{n_batches} batches")
    return DataLoader(
        ds, batch_size=gbs, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True,
        persistent_workers=True if cfg.num_workers > 0 else False)


def get_neg_embeddings_for_batch(batch_indices, hard_neg_map, cache,
                                  num_negs=4,
                                  device=torch.device("cuda")):
    B = len(batch_indices)
    D = cache.embeddings.shape[1]
    K = num_negs
    neg_embeds = torch.zeros(B, K, D)
    neg_mask = torch.zeros(B, K, dtype=torch.bool)
    for i, sample_idx in enumerate(batch_indices):
        if sample_idx in hard_neg_map:
            neg_cache_idxs = hard_neg_map[sample_idx]
            if len(neg_cache_idxs) > K:
                neg_cache_idxs = neg_cache_idxs[:K]
            k = len(neg_cache_idxs)
            neg_embeds[i, :k] = cache.lookup_indices(neg_cache_idxs)
            neg_mask[i, :k] = True
    return neg_embeds.to(device), neg_mask.to(device)


def plot_training(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(history["loss"], label="total")
    axes[0].plot(history["contrastive"], label="contrastive", alpha=0.7)
    axes[0].plot(history["triplet"], label="triplet", alpha=0.7)
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True)
    if history["R@10"]:
        axes[1].plot(history["eval_epochs"], history["R@10"], "o-",
                     label="R@10")
        axes[1].plot(history["eval_epochs"], history["R@50"], "s-",
                     label="R@50")
        axes[1].set_title("Recall"); axes[1].legend(); axes[1].grid(True)
    if history["lr"]:
        axes[2].plot(history["lr"]); axes[2].set_title("LR")
        axes[2].set_yscale("log"); axes[2].grid(True)
    plt.tight_layout(); plt.show()


print(" Utilities (fast gallery + CategoryBatchSampler)")


## Build model, data, optimizer, scheduler


In [ ]:
from torch.cuda.amp import autocast, GradScaler

set_seed(cfg.seed)
device = torch.device("cuda")

model = CIRModel(cfg).to(device)
processor = model.siglip_processor
resnet_tf = get_resnet_transform(cfg.resnet_input_size)

criterion = CIRLoss(
    margin=cfg.triplet_margin, temperature=cfg.temperature,
    lam_triplet=cfg.loss_lambda_triplet,
    lam_contrastive=cfg.loss_lambda_contrastive,
    label_smoothing=cfg.label_smoothing,
)

optimizer = torch.optim.AdamW(
    model.get_trainable_param_groups(), weight_decay=cfg.weight_decay)
scaler = GradScaler() if cfg.fp16 else None

print("Loading datasets")
train_datasets = [
    FashionIQDataset(
        cfg.fashioniq_root, cfg.image_dir, [cat], "train",
        processor, resnet_transform=resnet_tf,
    )
    for cat in cfg.categories
]
train_dataset = ConcatDataset(train_datasets)
offset = 0
for ds in train_datasets:
    ds.base_idx = offset
    offset += len(ds)

all_triplets = []
for ds in train_datasets:
    all_triplets.extend(ds.triplets)

print(f"\nCategory batch sampler (batch_size={cfg.batch_size}):")
cat_sampler = CategoryBatchSampler(
    train_dataset, cfg.categories,
    batch_size=cfg.batch_size, drop_last=True)

train_loader = DataLoader(
    train_dataset, batch_sampler=cat_sampler,
    num_workers=cfg.num_workers, pin_memory=True,
    collate_fn=collate_fn)

val_datasets = [
    FashionIQDataset(
        cfg.fashioniq_root, cfg.image_dir, [cat], "val",
        processor, resnet_transform=resnet_tf)
    for cat in cfg.categories
]
val_loader = DataLoader(
    ConcatDataset(val_datasets), batch_size=cfg.batch_size,
    shuffle=False, num_workers=cfg.num_workers,
    pin_memory=True, collate_fn=collate_fn)
val_gal_loaders = build_gallery_loaders(cfg, processor, resnet_tf, "val")

miner = HardNegativeMiner(
    cfg.num_hard_negatives, category_aware=True)
mining_gallery_loader = build_full_gallery_loader(
    cfg, processor, resnet_tf, "train")
train_id2cat = build_id2cat_mapping(cfg, split="train")

embed_cache = EmbeddingCache()
hard_neg_map = {}

total_steps = ((len(train_loader) + cfg.grad_accum_steps - 1)
               // cfg.grad_accum_steps) * cfg.num_epochs
warmup_steps = int(total_steps * cfg.warmup_ratio)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.01, 0.5 * (1 + np.cos(np.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

history = {"loss": [], "contrastive": [], "triplet": [],
           "R@10": [], "R@50": [], "eval_epochs": [], "lr": []}

best_recall = 0.0
global_step = 0


## Training

In [ ]:
print(f"Training: {cfg.num_epochs} epochs")
print(f"   Category batching + residual query")
print(f"  Batches/epoch: {len(train_loader)}")
print(f"  Effective BS: {cfg.batch_size} × {cfg.grad_accum_steps} = "
      f"{cfg.batch_size * cfg.grad_accum_steps}")
print(f"  Steps: {total_steps}, warmup: {warmup_steps}")
print(f"  Mining: every {cfg.mine_every_n_epochs} epoch, "
      f"fast=True")


for epoch in range(cfg.num_epochs):

    if epoch % cfg.mine_every_n_epochs == 0:
        embed_cache.build(model, mining_gallery_loader, device,
                          id2cat=train_id2cat)
        hard_neg_map = miner.mine(embed_cache, all_triplets)
        torch.cuda.empty_cache()
        gc.collect()
        import psutil
        ram = psutil.virtual_memory()
        print(f"   RAM: {ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB "
              f"({ram.percent}%)")

    model.train()

    epoch_loss = {"total": 0, "contrastive": 0, "triplet": 0}
    cat_counts = {}
    n_batches = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader,
                desc=f"Epoch {epoch+1}/{cfg.num_epochs}")
    for step, batch in enumerate(pbar):
        ref_px = batch["ref_pixel_values"].to(device)
        tgt_px = batch["target_pixel_values"].to(device)
        texts  = batch["mod_texts"]
        batch_idxs = batch["indices"]

        batch_cat = batch["categories"][0]
        cat_counts[batch_cat] = cat_counts.get(batch_cat, 0) + 1

        ref_rpx = batch.get("ref_resnet_pixels")
        tgt_rpx = batch.get("target_resnet_pixels")
        if ref_rpx is not None:
            ref_rpx = ref_rpx.to(device)
        if tgt_rpx is not None:
            tgt_rpx = tgt_rpx.to(device)

        neg_embeds, neg_mask = get_neg_embeddings_for_batch(
            batch_idxs, hard_neg_map, embed_cache,
            num_negs=cfg.num_neg_per_sample, device=device)

        if cfg.fp16:
            with autocast():
                query_emb  = model.encode_query(ref_px, texts, ref_rpx)
                target_emb = model.encode_target_train(tgt_px, tgt_rpx)
            losses = criterion(
                query_emb, target_emb,
                neg_embeds if neg_mask.any() else None,
                neg_mask   if neg_mask.any() else None)
            loss = losses["total"] / cfg.grad_accum_steps

            scaler.scale(loss).backward()
            if (step + 1) % cfg.grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
                global_step += 1
        else:
            query_emb  = model.encode_query(ref_px, texts, ref_rpx)
            target_emb = model.encode_target_train(tgt_px, tgt_rpx)
            losses = criterion(
                query_emb, target_emb,
                neg_embeds if neg_mask.any() else None,
                neg_mask   if neg_mask.any() else None)
            loss = losses["total"] / cfg.grad_accum_steps
            loss.backward()
            if (step + 1) % cfg.grad_accum_steps == 0:
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    cfg.max_grad_norm)
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
                global_step += 1

        for k in epoch_loss:
            epoch_loss[k] += losses[k].item()
        n_batches += 1
        pbar.set_postfix({
            "cat": batch_cat,
            "L": f"{losses['total'].item():.3f}",
            "T": f"{losses['triplet'].item():.3f}",
            "C": f"{losses['contrastive'].item():.3f}",
        })

    if n_batches % cfg.grad_accum_steps != 0:
        if cfg.fp16:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                cfg.max_grad_norm)
            optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
        global_step += 1

    for k in epoch_loss:
        epoch_loss[k] /= max(n_batches, 1)
    history["loss"].append(epoch_loss["total"])
    history["contrastive"].append(epoch_loss["contrastive"])
    history["triplet"].append(epoch_loss["triplet"])
    history["lr"].append(scheduler.get_last_lr()[0])

    cat_str = " | ".join(f"{c}={n}" for c, n in sorted(cat_counts.items()))
    print(f"  loss={epoch_loss['total']:.4f} | "
          f"cont={epoch_loss['contrastive']:.4f} | "
          f"trip={epoch_loss['triplet']:.4f} | "
          f"batches: {cat_str}")

    if (epoch + 1) % cfg.eval_every == 0 or epoch == cfg.num_epochs - 1:
        torch.cuda.empty_cache()
        metrics = evaluate_recall(
            model, val_loader, val_gal_loaders, cfg.recall_ks, device)
        for k, v in sorted(metrics.items()):
            print(f"    {k}: {v:.4f}")

        history["R@10"].append(metrics.get("avg_R@10", 0))
        history["R@50"].append(metrics.get("avg_R@50", 0))
        history["eval_epochs"].append(epoch + 1)

        r10 = metrics.get("avg_R@10", 0)
        if r10 > best_recall:
            best_recall = r10
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "metrics": metrics,
            }, f"{cfg.output_dir}/best_model.pt")
            print(f"     Best R@10={r10:.4f} saved!")

    if (epoch + 1) % 5 == 0:
        clear_output(wait=True)
        plot_training(history)

    torch.cuda.empty_cache()

print(f"\n Done! Best R@10: {best_recall:.4f}")
plot_training(history)

print(f"\n Done! Best R@10: {best_recall:.4f}")
plot_training(history)


## Load checkpoint and evaluate


In [ ]:
best_ckpt = torch.load(f"{cfg.output_dir}/best_model.pt",
                        map_location=device,weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"], strict=False)
print(f"Loaded best checkpoint (epoch {best_ckpt['epoch']}, "
      f"val R@10={best_ckpt['metrics'].get('avg_R@10', 0):.4f})")

test_datasets = [
    FashionIQDataset(
        cfg.fashioniq_root, cfg.image_dir, [cat], "val",
        processor, resnet_transform=resnet_tf,
    )
    for cat in cfg.categories
]

test_loader = DataLoader(
    ConcatDataset(test_datasets),
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    collate_fn=collate_fn,
)

test_gal_loaders = build_gallery_loaders(
    cfg, processor, resnet_tf, "val"
)

test_metrics = evaluate_recall(
    model, test_loader, test_gal_loaders, cfg.recall_ks, device
)

print("TEST RESULTS:")
for k, v in sorted(test_metrics.items()):
    print(f"  {k}: {v:.4f}")
print(f"  avg_R@10: {test_metrics.get('avg_R@10', 0):.4f}")
print(f"  avg_R@50: {test_metrics.get('avg_R@50', 0):.4f}")


## Checkpoint I/O

- **`save_checkpoint`** / **`load_checkpoint`**


In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, metrics, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": {
            k: v for k, v in model.state_dict().items()
            if "siglip" not in k
        },
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "metrics": metrics,
        "config": vars(cfg),
    }, path)
    print(f" Saved checkpoint: {path}")


def load_checkpoint(model, path, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location="cuda")
    model_dict = model.state_dict()
    pretrained = {k: v for k, v in ckpt["model_state_dict"].items() if k in model_dict}
    model_dict.update(pretrained)
    model.load_state_dict(model_dict, strict=False)
    print(f" Loaded {len(pretrained)} params from {path}")

    if optimizer and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler and "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    return ckpt.get("epoch", 0), ckpt.get("metrics", {})


save_checkpoint(
    model, optimizer, scheduler, cfg.num_epochs,
    {"best_R@10": best_recall}, f"{cfg.output_dir}/final_model.pt"
)


## Interactive retrieval

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
from PIL import Image
import os
@torch.no_grad()
def retrieve(model, processor, ref_image_path, mod_text,
             gallery_loader, resnet_transform=None, top_k=5, device=torch.device("cuda")):
    model.eval()

    img = Image.open(ref_image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt")
    ref_px = inputs["pixel_values"].to(device)

    resnet_px = None
    if resnet_transform is not None:
        resnet_px = resnet_transform(img).unsqueeze(0).to(device)

    q = model.encode_query(ref_px, [mod_text], resnet_px)

    all_embs, all_ids = [], []
    for batch in gallery_loader:
        if len(batch) == 3:
            pixels, r_pixels, ids = batch
            r_pixels = r_pixels.to(device)
        else:
            pixels, ids = batch
            r_pixels = None

        all_embs.append(model.encode_target(pixels.to(device), r_pixels).cpu())
        all_ids.extend(ids)
    all_embs = torch.cat(all_embs, 0)

    sims = F.cosine_similarity(q.cpu(), all_embs, dim=-1)
    topk_vals, topk_idx = sims.topk(top_k)

    return [(all_ids[i], topk_vals[j].item()) for j, i in enumerate(topk_idx)]


def show_retrieval(ref_path, mod_text, results, image_dir):
    n = len(results) + 1
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))

    axes[0].imshow(Image.open(ref_path))
    axes[0].set_title(f"Reference\n\"{mod_text}\"", fontsize=9)
    axes[0].axis("off")

    for i, (img_id, score) in enumerate(results):
        for ext in [".jpg", ".png"]:
            p = f"{image_dir}/{img_id}{ext}"
            if os.path.exists(p):
                axes[i+1].imshow(Image.open(p))
                break
        axes[i+1].set_title(f"#{i+1} (sim={score:.3f})\n{img_id}", fontsize=8)
        axes[i+1].axis("off")
    plt.tight_layout()
    plt.show()
